## 20 — Cited Papers Data Collection

For each award-winning paper we fetch its **reference list** (`referenced_works`) from OpenAlex,
then enrich every cited paper with the same paper-level attributes we already have for award papers.

**Workflow:**
1. Load `huang_matched_openalex.csv` → get unique award paper OpenAlex IDs
2. For each award paper: fetch `referenced_works` (list of cited paper IDs)
3. Build an edge table: `award_paper_id → cited_paper_id` (many-to-many handled via `cited_by` column)
4. Deduplicate cited paper IDs, batch-fetch full metadata from OpenAlex `/works`
5. Save to `data/matched/cited_papers.csv` and `data/matched/award_to_cited_edges.csv`

**Output files:**
- `cited_papers.csv` — one row per unique cited paper, same attributes as award papers
- `award_to_cited_edges.csv` — explicit mapping: which award paper cites which cited paper

In [1]:
import pandas as pd
import requests
import json
import time
from collections import defaultdict
from pathlib import Path

MAILTO   = 'shaheryar.4822@student.uu.se'
API_KEY  = 'A08hCjeUoeVKA9toVsfCpF'  # ← paste your key here
BASE_URL = 'https://api.openalex.org'

MATCHED_PATH    = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched\huang_matched_openalex.csv')
OUT_CITED       = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched\cited_papers.csv')
OUT_EDGES       = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched\award_to_cited_edges.csv')
CHECKPOINT_REFS = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched\cited_refs_checkpoint.csv')

def api_get(url, params=None):
    p = {'mailto': MAILTO, 'api_key': API_KEY}
    if params:
        p.update(params)
    for attempt in range(3):
        try:
            r = requests.get(url, params=p, timeout=15)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:
                time.sleep(5)
        except requests.RequestException:
            time.sleep(2)
    return None

print('Ready.')

Ready.


### Step 1 — Load award papers & get unique OpenAlex IDs

In [2]:
df = pd.read_csv(MATCHED_PATH)

# Confirm we're only using 2000-2018
df = df[df['year'].between(2000, 2018)].copy()

# Drop rows with no openalex_id
df = df.dropna(subset=['openalex_id'])

# One row per unique award paper (paper can appear multiple times for multi-author rows)
award_papers = df.drop_duplicates(subset='openalex_id')[['openalex_id', 'year', 'conference', 'paper_title']].copy()
award_papers['openalex_id'] = award_papers['openalex_id'].str.strip()

print(f'Total award papers (2000-2018): {len(award_papers)}')
print(f'Year range: {award_papers["year"].min()} – {award_papers["year"].max()}')
print(award_papers.head(3))

Total award papers (2000-2018): 890
Year range: 2000 – 2018
                        openalex_id  year conference  \
0  https://openalex.org/W2788603415  2018       AAAI   
1  https://openalex.org/W2798397965  2018        ACL   
2  https://openalex.org/W2963033005  2018        ACL   

                                         paper_title  
0           Memory-Augmented Monte Carlo Tree Search  
1  Finding syntax in human encephalography with b...  
2  Learning to Ask Good Questions: Ranking Clarif...  


### Step 2 — Fetch `referenced_works` for each award paper

In [3]:
# Resume from checkpoint if it exists
if CHECKPOINT_REFS.exists():
    edges_df = pd.read_csv(CHECKPOINT_REFS)
    done_ids = set(edges_df['award_paper_id'].unique())
    edges = edges_df.to_dict('records')
    print(f'Resuming — {len(done_ids)} award papers already processed')
else:
    edges = []
    done_ids = set()

to_process = award_papers[~award_papers['openalex_id'].isin(done_ids)]
print(f'Award papers left to process: {len(to_process)}')

for i, row in enumerate(to_process.itertuples(), 1):
    oa_id = row.openalex_id
    short_id = oa_id.split('/')[-1]

    data = api_get(f'{BASE_URL}/works/{short_id}', params={'select': 'id,referenced_works'})
    time.sleep(0.12)

    if not data:
        print(f'  [{i}] ✗ No data for {short_id}')
        continue

    refs = data.get('referenced_works', [])

    for cited_id in refs:
        edges.append({
            'award_paper_id':    oa_id,
            'award_year':        row.year,
            'award_conference':  row.conference,
            'cited_paper_id':    cited_id
        })

    if i % 50 == 0:
        pd.DataFrame(edges).to_csv(CHECKPOINT_REFS, index=False)
        print(f'  [{i}/{len(to_process)}] checkpoint saved — {len(edges)} edges so far')

edges_df = pd.DataFrame(edges)
edges_df.to_csv(CHECKPOINT_REFS, index=False)

print(f'\nTotal citation edges: {len(edges_df)}')
print(f'Unique cited paper IDs: {edges_df["cited_paper_id"].nunique()}')

Award papers left to process: 890
  [50/890] checkpoint saved — 2682 edges so far
  [100/890] checkpoint saved — 4964 edges so far
  [150/890] checkpoint saved — 7627 edges so far
  [200/890] checkpoint saved — 10142 edges so far
  [250/890] checkpoint saved — 12176 edges so far
  [300/890] checkpoint saved — 14204 edges so far
  [350/890] checkpoint saved — 16517 edges so far
  [400/890] checkpoint saved — 18225 edges so far
  [450/890] checkpoint saved — 20026 edges so far
  [500/890] checkpoint saved — 21662 edges so far
  [550/890] checkpoint saved — 23425 edges so far
  [600/890] checkpoint saved — 25442 edges so far
  [650/890] checkpoint saved — 27286 edges so far
  [700/890] checkpoint saved — 28867 edges so far
  [750/890] checkpoint saved — 30381 edges so far
  [800/890] checkpoint saved — 32004 edges so far
  [850/890] checkpoint saved — 33392 edges so far

Total citation edges: 34417
Unique cited paper IDs: 29427


### Step 3 — Build `cited_by` lookup & deduplicate cited IDs

In [4]:
# For each cited paper, collect the list of award papers that cite it
cited_by_map = (
    edges_df.groupby('cited_paper_id')['award_paper_id']
    .apply(lambda x: '|'.join(sorted(x.unique())))
    .reset_index()
    .rename(columns={'award_paper_id': 'cited_by_award_papers'})
)

cited_by_count = (
    edges_df.groupby('cited_paper_id')['award_paper_id']
    .nunique()
    .reset_index()
    .rename(columns={'award_paper_id': 'cited_by_n_award_papers'})
)

cited_meta = cited_by_map.merge(cited_by_count, on='cited_paper_id')
unique_cited_ids = cited_meta['cited_paper_id'].tolist()

print(f'Unique cited papers to enrich: {len(unique_cited_ids)}')
print(f'Cited by multiple award papers (>=2): {(cited_meta["cited_by_n_award_papers"] >= 2).sum()}')

Unique cited papers to enrich: 29427
Cited by multiple award papers (>=2): 3312


### Step 4 — Batch-fetch cited paper metadata from OpenAlex
OpenAlex supports filter OR batches of up to 100 IDs at a time.

In [ ]:
def extract_paper_attrs(w):
    """Pull the same attributes we have for award papers."""
    # Primary location / venue
    primary_loc = w.get('primary_location') or {}
    source      = primary_loc.get('source') or {}

    # Best OA url
    best_oa = w.get('best_oa_location') or {}

    # Authors
    authorships = w.get('authorships', [])
    author_ids   = '|'.join([a['author']['id'] for a in authorships if a.get('author') and a['author'].get('id')])
    author_names = '|'.join([a['author'].get('display_name','') for a in authorships if a.get('author') and a['author'].get('id')])
    author_positions = '|'.join([a.get('author_position','') for a in authorships])

    # Institutions (first author's)
    first_insts = []
    if authorships:
        first_insts = [i.get('display_name','') for i in authorships[0].get('institutions', [])]

    # Topics / concepts
    topics   = w.get('topics', [])
    top_topic = topics[0].get('display_name','') if topics else ''
    top_field = topics[0].get('field', {}).get('display_name','') if topics else ''

    # Counts by year (citations per year)
    counts_by_year = w.get('counts_by_year', [])

    return {
        'openalex_id':          w.get('id',''),
        'doi':                  w.get('doi',''),
        'title':                w.get('title',''),
        'publication_year':     w.get('publication_year'),
        'publication_date':     w.get('publication_date',''),
        'type':                 w.get('type',''),
        'cited_by_count':       w.get('cited_by_count', 0),
        'is_retracted':         w.get('is_retracted', False),
        'is_oa':                w.get('open_access', {}).get('is_oa', False),
        'source_id':            source.get('id',''),
        'source_name':          source.get('display_name',''),
        'source_type':          source.get('type',''),
        'source_issn':          '|'.join(source.get('issn', []) or []),
        'author_ids':           author_ids,
        'author_names':         author_names,
        'author_positions':     author_positions,
        'author_count':         len(authorships),
        'first_author_institution': '|'.join(first_insts),
        'top_topic':            top_topic,
        'top_field':            top_field,
        'counts_by_year':       json.dumps(counts_by_year),
    }


def fetch_works_batch(id_list):
    """Fetch up to 100 works in one request using filter OR syntax."""
    ids_str = '|'.join([i.split('/')[-1] for i in id_list])
    data = api_get(
        f'{BASE_URL}/works',
        params={
            'filter': f'openalex_id:{ids_str}',
            'per-page': 100,
            'select': (
                'id,doi,title,publication_year,publication_date,type,'
                'cited_by_count,is_retracted,open_access,primary_location,'
                'best_oa_location,authorships,topics,counts_by_year'
            )
        }
    )
    if data:
        return data.get('results', [])
    return []


# Check for existing progress
CHECKPOINT_PAPERS = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched\cited_papers_checkpoint.csv')

if CHECKPOINT_PAPERS.exists():
    existing = pd.read_csv(CHECKPOINT_PAPERS)
    fetched_ids = set(existing['openalex_id'].tolist())
    all_papers = existing.to_dict('records')
    print(f'Resuming — {len(fetched_ids)} cited papers already fetched')
else:
    fetched_ids = set()
    all_papers = []

remaining = [i for i in unique_cited_ids if i not in fetched_ids]
print(f'Cited papers left to fetch: {len(remaining)}')

BATCH = 100
batches = [remaining[i:i+BATCH] for i in range(0, len(remaining), BATCH)]

for b_idx, batch in enumerate(batches, 1):
    results = fetch_works_batch(batch)
    time.sleep(0.15)

    for w in results:
        all_papers.append(extract_paper_attrs(w))

    if b_idx % 50 == 0:
        pd.DataFrame(all_papers).to_csv(CHECKPOINT_PAPERS, index=False)
        pct = b_idx / len(batches) * 100
        print(f'  [batch {b_idx}/{len(batches)} | {pct:.1f}%] {len(all_papers)} papers fetched')

cited_papers_df = pd.DataFrame(all_papers)
cited_papers_df.to_csv(CHECKPOINT_PAPERS, index=False)
print(f'\nTotal cited papers fetched: {len(cited_papers_df)}')

Cited papers left to fetch: 29427


TypeError: sequence item 0: expected str instance, NoneType found

### Step 5 — Merge `cited_by` info & save final outputs

In [ ]:
# Join the cited_by metadata onto the papers dataframe
final_cited = cited_papers_df.merge(cited_meta, left_on='openalex_id', right_on='cited_paper_id', how='left')
final_cited = final_cited.drop(columns=['cited_paper_id'], errors='ignore')

# Save cited papers (one row per unique cited paper)
final_cited.to_csv(OUT_CITED, index=False)

# Save the edge table (explicit award → cited mapping)
edges_df.to_csv(OUT_EDGES, index=False)

print('=' * 60)
print(f'cited_papers.csv        → {len(final_cited)} rows')
print(f'award_to_cited_edges.csv → {len(edges_df)} rows')
print(f'\ncited_by_n_award_papers distribution:')
print(final_cited['cited_by_n_award_papers'].value_counts().sort_index().head(10))
print(f'\nMissing cited_by (not found in OpenAlex): {final_cited["cited_by_award_papers"].isna().sum()}')
print(f'\nYear coverage of cited papers:')
print(final_cited['publication_year'].value_counts().sort_index().head(20))
print(f'\nSample row:')
print(final_cited[['openalex_id','title','publication_year','cited_by_count',
                    'cited_by_n_award_papers','cited_by_award_papers']].head(3).to_string())